# Lesson 21 — 资源所有权、Mutex 与何时不用 RTOS — Solution

这是独立答案 Notebook。先完成主课再打开它。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

In [ ]:
%%academy_lab 21-resource-ownership solution exercise pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device& device) : device_(device) {
        }
        void tick();

    private:
        Device& device_;
        bool previous_button_ = false;
        int pending_notifications_ = 0;
    };
}

namespace academy {
    void Controller::tick() {
        if (device_.input.button && !previous_button_)
            ++pending_notifications_;
        previous_button_ = device_.input.button;
        device_.output.error = device_.input.fault;
        if (device_.input.fault)
            device_.output.state = "fault";
        else if (!device_.input.connected)
            device_.output.state = "waiting-link";
        else if (pending_notifications_ > 0) {
            --pending_notifications_;
            ++device_.output.events;
            device_.output.outgoing = "notification";
            device_.output.state = "sent";
        } else
            device_.output.state = "idle";
        device_.output.queue_depth = pending_notifications_;
    }
}
